# 05 — Recuperar y registrar textos completos

Esta fase toma las fuentes `include` y `uncertain` del cribado por título y resumen y construye una cola auditable de recuperación de texto completo.

Principios operativos:

- una página editorial no equivale a texto completo;
- solo se descargan PDF identificados explícitamente como acceso abierto;
- no se evaden paywalls, autenticación ni controles técnicos;
- cada archivo local se registra con URL, fecha, tamaño, tipo MIME y SHA-256;
- las fuentes `uncertain` permanecen en una cola separada para cribado a texto completo;
- la resolución OpenAlex y la descarga están desactivadas por defecto y requieren activación explícita.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.full_text import (
    build_full_text_manifest,
    build_retrieval_queue,
    download_open_access_queue,
    initialise_retrieval_sheet,
    load_full_text_config,
    read_csv_robust,
    register_local_files,
    resolve_openalex_queue,
    retrieval_summary,
    split_retrieval_outputs,
    validate_retrieval_sheet,
)

CONFIG_PATH = ROOT / "config" / "full_text_retrieval.yml"
config = load_full_text_config(CONFIG_PATH)

INTERIM = ROOT / "data" / "interim"
FULL_TEXT_DIR = ROOT / config["paths"]["full_text_dir"]
INTERIM.mkdir(parents=True, exist_ok=True)
FULL_TEXT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Full-text directory: {FULL_TEXT_DIR.relative_to(ROOT)}")


## 1. Cargar las decisiones de la fase 04

La cola debe contener todas las fuentes incluidas y todas las inciertas. Las fuentes excluidas no avanzan.


In [ ]:
included_path = ROOT / config["paths"]["included_csv"]
uncertain_path = ROOT / config["paths"]["uncertain_csv"]

missing_inputs = [
    str(path.relative_to(ROOT))
    for path in (included_path, uncertain_path)
    if not path.exists()
]
if missing_inputs:
    raise FileNotFoundError(
        "Faltan salidas de la fase 04: " + ", ".join(missing_inputs)
    )

included, included_encoding = read_csv_robust(included_path)
uncertain, uncertain_encoding = read_csv_robust(uncertain_path)

print(f"Included: {len(included)} ({included_encoding})")
print(f"Uncertain: {len(uncertain)} ({uncertain_encoding})")
print(f"Expected retrieval queue: {len(included) + len(uncertain)}")


## 2. Crear o actualizar la hoja de recuperación

Las columnas de metadatos se reconstruyen desde las decisiones de cribado. Las búsquedas, notas, rutas locales y estados previamente registrados se preservan mediante `source_id`.


In [ ]:
queue = build_retrieval_queue(included, uncertain)

working_path = ROOT / config["paths"]["working_csv"]
existing = (
    read_csv_robust(working_path)[0]
    if working_path.exists()
    else pd.DataFrame()
)

working = initialise_retrieval_sheet(queue, existing)
working.to_csv(working_path, index=False, encoding="utf-8-sig")

print(f"Queue rows: {len(working)}")
print(f"Unique source_id: {working['source_id'].nunique()}")
print(f"Saved: {working_path.relative_to(ROOT)}")
display(retrieval_summary(working))


## 3. Registrar archivos locales existentes

Un archivo local debe comenzar con su `source_id`, por ejemplo:

```text
src_123456789abc__short-title.pdf
```

También se acepta `src_123456789abc.pdf`. Cuando existen varios archivos para una misma fuente, se registra el primero y se genera una advertencia.


In [ ]:
working, local_errors = register_local_files(
    working,
    FULL_TEXT_DIR,
    project_root=ROOT,
)

working.to_csv(working_path, index=False, encoding="utf-8-sig")

print(f"Registered local files: {(working['retrieval_status'] == 'available_local').sum()}")
print(f"Local-registration errors: {len(local_errors)}")
display(local_errors.head(20))


## 4. Resolver DOI y acceso abierto con OpenAlex

Esta celda está desactivada por defecto. Para ejecutarla:

1. define `OPENALEX_API_KEY` en tu entorno;
2. cambia `RESOLVE_OPENALEX = True`;
3. ejecuta únicamente esta celda.

La consulta recupera metadatos de acceso y una URL PDF cuando OpenAlex identifica una localización de acceso abierto. No descarga todavía ningún documento.


In [ ]:
RESOLVE_OPENALEX = False

openalex_errors = pd.DataFrame(
    columns=[
        "source_id",
        "stage",
        "error_type",
        "error_message",
    ]
)

if RESOLVE_OPENALEX:
    key_name = config["openalex"]["api_key_env"]
    api_key = os.getenv(key_name, "").strip()

    if not api_key:
        raise RuntimeError(
            f"Falta la variable de entorno {key_name}. "
            "La clave debe configurarse antes de iniciar JupyterLab."
        )

    working, openalex_errors = resolve_openalex_queue(
        working,
        api_key=api_key,
        config=config,
        overwrite=bool(
            config["openalex"]["overwrite_resolved"]
        ),
    )

    working.to_csv(
        working_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("OpenAlex resolution completed.")

else:
    print(
        "OpenAlex resolution disabled. "
        "Set RESOLVE_OPENALEX = True to run it."
    )

display(retrieval_summary(working))
display(openalex_errors.head(20))

In [ ]:
display(
    working["openalex_lookup_status"]
    .replace("", "not_attempted")
    .value_counts(dropna=False)
    .rename_axis("openalex_lookup_status")
    .reset_index(name="sources")
)

display(
    working["retrieval_status"]
    .replace("", "pending")
    .value_counts(dropna=False)
    .rename_axis("retrieval_status")
    .reset_index(name="sources")
)

print(
    "PDF abiertos identificados:",
    int(working["retrieval_status"].eq("available_open_access").sum())
)

print(
    "Errores OpenAlex:",
    len(openalex_errors)
)

display(
    working.loc[
        working["retrieval_status"].eq("available_open_access"),
        [
            "source_id",
            "title",
            "doi",
            "oa_status",
            "license",
            "best_pdf_url",
        ],
    ]
)

## 5. Descargar PDF de acceso abierto

Esta celda también está desactivada por defecto. Solo procesa filas con:

```text
retrieval_status = available_open_access
is_open_access = true
best_pdf_url = URL directa
```

Cada descarga se limita por tamaño, valida la firma `%PDF-` y se registra con SHA-256. Los errores quedan auditados sin detener toda la cola.


In [ ]:
DOWNLOAD_OPEN_ACCESS_PDFS = False

download_errors = pd.DataFrame(
    columns=["source_id", "stage", "error_type", "error_message"]
)

if DOWNLOAD_OPEN_ACCESS_PDFS:
    working, download_errors = download_open_access_queue(
        working,
        FULL_TEXT_DIR,
        config=config,
        overwrite=bool(config["download"]["overwrite_existing"]),
    )

    working, rescan_errors = register_local_files(
        working,
        FULL_TEXT_DIR,
        project_root=ROOT,
    )
    local_errors = pd.concat(
        [local_errors, rescan_errors],
        ignore_index=True,
    ).drop_duplicates()
    working.to_csv(working_path, index=False, encoding="utf-8-sig")
    print("Open-access download completed.")
else:
    print(
        "PDF download disabled. "
        "Resolve OpenAlex first and set DOWNLOAD_OPEN_ACCESS_PDFS = True."
    )

display(retrieval_summary(working))
display(download_errors.head(20))


In [ ]:
display(
    working["openalex_lookup_status"]
    .replace("", "not_attempted")
    .value_counts(dropna=False)
    .rename_axis("openalex_lookup_status")
    .reset_index(name="sources")
)

display(
    working["retrieval_status"]
    .replace("", "pending")
    .value_counts(dropna=False)
    .rename_axis("retrieval_status")
    .reset_index(name="sources")
)

print(
    "PDF abiertos identificados:",
    int(working["retrieval_status"].eq("available_open_access").sum())
)

print(
    "Errores OpenAlex:",
    len(openalex_errors)
)

display(
    working.loc[
        working["retrieval_status"].eq("available_open_access"),
        [
            "source_id",
            "title",
            "doi",
            "oa_status",
            "license",
            "best_pdf_url",
        ],
    ]
)

## 6. Asignar acciones manuales pendientes

Estas acciones no alteran la decisión de elegibilidad. Organizan el trabajo pendiente para biblioteca, verificación de metadatos o cribado a texto completo.


In [ ]:
action_map = {
    "restricted_access": "obtain_through_library",
    "landing_page_only": "verify_landing_page",
    "not_found": "resolve_metadata",
    "retrieval_error": "resolve_metadata",
}

blank_action = working["manual_action"].astype(str).str.strip().eq("")
for status, action in action_map.items():
    mask = blank_action & working["retrieval_status"].eq(status)
    working.loc[mask, "manual_action"] = action

uncertain_mask = (
    working["screening_decision"].eq("uncertain")
    & working["manual_action"].astype(str).str.strip().eq("")
)
working.loc[uncertain_mask, "manual_action"] = "full_text_screening"

working.to_csv(working_path, index=False, encoding="utf-8-sig")

display(
    working["manual_action"]
    .replace("", "none")
    .value_counts()
    .rename_axis("manual_action")
    .reset_index(name="sources")
)


## 7. Validar la hoja de recuperación

La validación comprueba identificadores duplicados, estados desconocidos, PDF abiertos sin URL y archivos locales inexistentes.


In [ ]:
working, detected_encoding = read_csv_robust(working_path)

issues = validate_retrieval_sheet(
    working,
    config,
    project_root=ROOT,
)

issues_path = ROOT / config["paths"]["issues_csv"]
issues.to_csv(issues_path, index=False, encoding="utf-8-sig")

print(f"Encoding: {detected_encoding}")
print(f"Validation issues: {len(issues)}")
display(issues.head(50))
display(retrieval_summary(working))


## 8. Exportar manifiesto, colas y auditoría

- `full_text_manifest.csv`: archivos locales registrados;
- `full_text_available.csv`: texto local o PDF abierto identificado;
- `full_text_unavailable.csv`: acceso restringido, página editorial, no encontrado o error;
- `full_text_screening_queue.csv`: las fuentes `uncertain`;
- `full_text_pending.csv`: filas aún sin estado;
- `full_text_retrieval_errors.csv`: errores de resolución, descarga o registro.


In [ ]:
manifest = build_full_text_manifest(working)
groups = split_retrieval_outputs(working)

errors = pd.concat(
    [local_errors, openalex_errors, download_errors],
    ignore_index=True,
).drop_duplicates()

output_frames = {
    config["paths"]["manifest_csv"]: manifest,
    config["paths"]["available_csv"]: groups["available"],
    config["paths"]["unavailable_csv"]: groups["unavailable"],
    config["paths"]["screening_queue_csv"]: groups["screening_queue"],
    config["paths"]["pending_csv"]: groups["pending"],
    config["paths"]["errors_csv"]: errors,
}

for relative_path, frame in output_frames.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(retrieval_summary(working))


## Criterio para avanzar a la fase 06

La extracción y segmentación de documentos comienza cuando:

1. la hoja de recuperación no presenta errores estructurales;
2. cada archivo local tiene `source_id`, ruta, tamaño, tipo MIME y SHA-256;
3. las fuentes sin texto completo tienen una acción manual explícita;
4. las fuentes `uncertain` están identificadas en la cola de cribado a texto completo;
5. se conserva una copia inalterada de cada documento recuperado.

La fase siguiente será:

```text
notebooks/06_parse_and_segment_documents.ipynb
```
